In [ ]:
import pandas as pd
df=pd.read_json("D:/vuln-detection-ml/data/function.json")
print(df.shape)
print(df.columns.tolist())
df.head()

(27318, 4)
['project', 'commit_id', 'target', 'func']


,project,commit_id,target,func
0,FFmpeg,973b1a6b9070e2bf17d17568cbaf4043ce931f51,0,static av_cold int vdadec_init(AVCodecContext ...
1,FFmpeg,321b2a9ded0468670b7678b7c098886930ae16b2,0,static int transcode(AVFormatContext **output_...
2,FFmpeg,5d5de3eba4c7890c2e8077f5b4ae569671d11cf8,0,"static void v4l2_free_buffer(void *opaque, uin..."
3,FFmpeg,32bf6550cb9cc9f487a6722fe2bfc272a93c1065,0,"int ff_get_wav_header(AVFormatContext *s, AVIO..."
4,FFmpeg,57d77b3963ce1023eaf5ada8cba58b9379405cc8,0,"int av_opencl_buffer_write(cl_mem dst_cl_buf, ..."


In [3]:
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True))

target
0    14858
1    12460
Name: count, dtype: int64
target
0    0.54389
1    0.45611
Name: proportion, dtype: float64


In [ ]:
df["length"] = df["func"].str.len()
print(df["length"].describe())

count     27318.000000
mean       2023.964712
std        4531.181073
min          26.000000
25%         404.250000
50%         937.000000
75%        2176.750000
max      142346.000000
Name: length, dtype: float64


In [5]:
print(df["project"].value_counts())

project
qemu      17549
FFmpeg     9769
Name: count, dtype: int64


In [6]:
print(df[df["target"] == 1]["func"].iloc[0])

static int r3d_read_rdvo(AVFormatContext *s, Atom *atom)

{

    R3DContext *r3d = s->priv_data;

    AVStream *st = s->streams[0];

    int i;



    r3d->video_offsets_count = (atom->size - 8) / 4;

    r3d->video_offsets = av_malloc(atom->size);

    if (!r3d->video_offsets)

        return AVERROR(ENOMEM);



    for (i = 0; i < r3d->video_offsets_count; i++) {

        r3d->video_offsets[i] = avio_rb32(s->pb);

        if (!r3d->video_offsets[i]) {

            r3d->video_offsets_count = i;

            break;

        }

        av_dlog(s, "video offset %d: %#x\n", i, r3d->video_offsets[i]);

    }



    if (st->r_frame_rate.num)

        st->duration = av_rescale_q(r3d->video_offsets_count,

                                    (AVRational){st->r_frame_rate.den,

                                                 st->r_frame_rate.num},

                                    st->time_base);

    av_dlog(s, "duration %"PRId64"\n", st->duration);



    return 0;

}



generating train, validation and test data

In [ ]:
from sklearn.model_selection import train_test_split
train, temp = train_test_split(df,test_size=0.2, random_state=42, stratify=df["target"])
valid, test = train_test_split(temp,test_size=0.5,random_state=42, stratify=temp["target"])
print(len(train), len(valid), len(test))
train.to_json("D:/vuln-detection-ml/data/train.jsonl",orient="records",lines=True)
valid.to_json("D:/vuln-detection-ml/data/valid.jsonl",orient="records",lines=True)
test.to_json("D:/vuln-detection-ml/data/test.jsonl",orient="records",lines=True)


21854 2732 2732
